# AVERT — Evaluation & Analysis Notebook
Compares curriculum-trained agent vs scratch baseline across:
- Stage 3 (trained environment)
- Medium-difficulty highway (generalization probe)
- Training efficiency curves
- Transfer learning (roundabout / merge)

Every agent is rendered as a GIF for each environment.

In [ ]:
# ── Cell 1: system deps ───────────────────────────────────────────
!apt-get install -y xvfb > /dev/null 2>&1
!pip install highway-env stable-baselines3 gymnasium pyvirtualdisplay imageio -q

In [ ]:
# ── Cell 2: virtual display ───────────────────────────────────────
from pyvirtualdisplay import Display
vdisplay = Display(visible=False, size=(600, 200))
vdisplay.start()
print("Display alive:", vdisplay.is_alive())

In [ ]:
# ── Cell 3: imports + drive ───────────────────────────────────────
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import imageio
import gymnasium as gym
import highway_env
from IPython.display import Image, display
from stable_baselines3 import PPO
from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/AVERT/"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Drive mounted. SAVE_DIR:", SAVE_DIR)

In [ ]:
# ── Cell 4: custom vehicle + all configs ─────────────────────────
import highway_env.vehicle.behavior as behavior
from highway_env.vehicle.behavior import IDMVehicle

class PursuerVehicle(IDMVehicle):
    """Actively chases the ego vehicle by matching its lane and flooring it."""
    PURSUIT_GAIN = 1.5
    def act(self, action=None):
        ego = self.road.vehicles[0]
        if self.lane_index[2] < ego.lane_index[2]:
            action = "LANE_RIGHT"
        elif self.lane_index[2] > ego.lane_index[2]:
            action = "LANE_LEFT"
        else:
            action = "FASTER"
        super().act(action)

behavior.PursuerVehicle = PursuerVehicle

stage1_config = {
    "observation": {"type": "Kinematics"},
    "action": {"type": "DiscreteMetaAction"},
    "lanes_count": 4,
    "vehicles_count": 30,
    "duration": 40,
    "other_vehicles_type": "highway_env.vehicle.behavior.IDMVehicle",
    "collision_reward": -2,
    "high_speed_reward": 0.4,
    "right_lane_reward": 0.1,
    "lane_change_reward": 0.0,
    "reward_speed_range": [20, 30],
    "normalize_reward": True,
}

stage2_config = {
    **stage1_config,
    "vehicles_count": 40,
    "other_vehicles_type": "highway_env.vehicle.behavior.AggressiveVehicle",
    "collision_reward": -3,
}

stage3_config = {
    **stage1_config,
    "vehicles_count": 50,
    "other_vehicles_type": "highway_env.vehicle.behavior.PursuerVehicle",
    "collision_reward": -5,
}

# Medium: same pursuer type as Stage 3 but fewer cars + softer collision penalty
# Difficulty sits between Stage 2 and Stage 3
medium_config = {
    **stage1_config,
    "vehicles_count": 40,
    "other_vehicles_type": "highway_env.vehicle.behavior.PursuerVehicle",
    "collision_reward": -3,
}

roundabout_config = {
    "observation": {"type": "Kinematics"},
    "action": {"type": "DiscreteMetaAction"},
    "collision_reward": -3,
    "high_speed_reward": 0.4,
    "normalize_reward": True,
}

merge_config = {
    "observation": {"type": "Kinematics"},
    "action": {"type": "DiscreteMetaAction"},
    "collision_reward": -3,
    "high_speed_reward": 0.4,
    "normalize_reward": True,
}

print("All configs ready.")

In [ ]:
# ── Cell 5: helper functions ──────────────────────────────────────

def render_agent(model, config, env_id="highway-v0", filename="agent.gif", n_episodes=3):
    """Render `n_episodes`, save a GIF to Drive, display inline, and print stats."""
    env = gym.make(env_id, render_mode="rgb_array")
    env.unwrapped.configure(config)
    env.unwrapped.configure({"offscreen_rendering": True})
    frames, rewards, crashed_eps = [], [], []

    for ep in range(n_episodes):
        obs, _ = env.reset()
        env.step(env.action_space.sample())  # init viewer
        ep_reward, done, truncated, ep_crashed = 0.0, False, False, False
        while not (done or truncated):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, truncated, info = env.step(action)
            frame = env.render()
            if frame is not None:
                frames.append(frame)
            ep_reward += reward
            if info.get("crashed", False):
                ep_crashed = True
        rewards.append(ep_reward)
        crashed_eps.append(ep_crashed)
        crash_tag = "CRASH" if ep_crashed else "survived"
        print(f"  Episode {ep+1}: reward={ep_reward:.2f}  [{crash_tag}]")

    env.close()
    if frames:
        save_path = os.path.join(SAVE_DIR, filename)
        imageio.mimsave(save_path, frames, fps=15)
        display(Image(save_path))
    survival = sum(1 for c in crashed_eps if not c) / n_episodes
    print(f"  Avg reward: {np.mean(rewards):.2f} | Survival: {survival*100:.0f}%")


def eval_agent(model, config, env_id="highway-v0", n_episodes=10, seed=42):
    """Headless evaluation returning reward/crash stats (no frames saved)."""
    env = gym.make(env_id, render_mode=None)
    env.unwrapped.configure(config)
    rewards, crashed_flags = [], []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        ep_reward, done, truncated, ep_crashed = 0.0, False, False, False
        while not (done or truncated):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, truncated, info = env.step(action)
            ep_reward += reward
            if info.get("crashed", False):
                ep_crashed = True
        rewards.append(ep_reward)
        crashed_flags.append(ep_crashed)
    env.close()
    survival = sum(1 for c in crashed_flags if not c) / n_episodes
    return {
        "rewards": rewards, "crashed": crashed_flags,
        "survival_rate": survival,
        "mean_reward": np.mean(rewards),
        "std_reward": np.std(rewards),
    }


def smooth(x, w=5):
    if len(x) < w:
        return np.array(x, dtype=float)
    return np.convolve(x, np.ones(w) / w, mode='valid')

print("Helper functions ready.")

In [ ]:
# ── Cell 6: load all models ───────────────────────────────────────
print("Loading models from Drive...")

# Curriculum checkpoints at each stage
model_stage1     = PPO.load(os.path.join(SAVE_DIR, "avert_stage1_final"))
model_stage2     = PPO.load(os.path.join(SAVE_DIR, "avert_stage2_final"))
model_curriculum = PPO.load(os.path.join(SAVE_DIR, "avert_stage3_final"))

# Scratch baseline trained directly on Stage 3
model_scratch = PPO.load(os.path.join(SAVE_DIR, "avert_stage3_scratch_final"))

# Curriculum fine-tuned on new environments
model_roundabout_transfer = PPO.load(os.path.join(SAVE_DIR, "avert_roundabout_transfer_final"))
model_merge_transfer      = PPO.load(os.path.join(SAVE_DIR, "avert_merge_transfer_final"))

# Scratch baselines trained from scratch on those same environments
model_roundabout_scratch = PPO.load(os.path.join(SAVE_DIR, "avert_roundabout_scratch_final"))
model_merge_scratch      = PPO.load(os.path.join(SAVE_DIR, "avert_merge_scratch_final"))

print("All 8 models loaded.")

---
## 1 · Curriculum Agent — Stage-by-Stage Renders
Shows how the curriculum agent performs on each of its training environments,  
letting you see qualitative improvement as it progresses through harder stages.

In [ ]:
# ── Stage 1 model on Stage 1 env (passive IDM traffic) ───────────
print("=== Curriculum Stage 1 Agent — Stage 1 Config (30 IDM cars) ===")
render_agent(model_stage1, stage1_config, env_id="highway-v0",
             filename="curr_stage1_on_stage1.gif", n_episodes=3)

In [ ]:
# ── Stage 2 model on Stage 2 env (aggressive traffic) ────────────
print("=== Curriculum Stage 2 Agent — Stage 2 Config (40 Aggressive cars) ===")
render_agent(model_stage2, stage2_config, env_id="highway-v0",
             filename="curr_stage2_on_stage2.gif", n_episodes=3)

In [ ]:
# ── Stage 3 model on Stage 3 env (active pursuers) ───────────────
print("=== Curriculum Stage 3 Agent — Stage 3 Config (50 PursuerVehicles) ===")
render_agent(model_curriculum, stage3_config, env_id="highway-v0",
             filename="curr_stage3_on_stage3.gif", n_episodes=3)

---
## 2 · Stage 3 Head-to-Head: Curriculum vs Scratch
Both agents tested on the same Stage 3 environment (50 PursuerVehicles, collision_reward=–5).

In [ ]:
# ── Quantitative eval (15 episodes, no rendering) ────────────────
N_EVAL = 15
print(f"Evaluating both agents on Stage 3 ({N_EVAL} episodes each, headless)...")

stats_curr    = eval_agent(model_curriculum, stage3_config, n_episodes=N_EVAL)
stats_scratch = eval_agent(model_scratch,    stage3_config, n_episodes=N_EVAL)

print("\n── Stage 3 Results ─────────────────────────────")
for label, s in [("Curriculum", stats_curr), ("Scratch", stats_scratch)]:
    print(f"  {label:12s}  mean={s['mean_reward']:6.2f}  std={s['std_reward']:5.2f}"
          f"  survival={s['survival_rate']*100:.0f}%")

In [ ]:
# ── Bar chart: Stage 3 mean reward + survival ─────────────────────
labels = ["Curriculum\n(Stage 1→2→3)", "Scratch\n(Stage 3 only)"]
means  = [stats_curr['mean_reward'],    stats_scratch['mean_reward']]
stds   = [stats_curr['std_reward'],     stats_scratch['std_reward']]
surv   = [stats_curr['survival_rate']*100, stats_scratch['survival_rate']*100]
colors = ["steelblue", "crimson"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

bars = axes[0].bar(labels, means, yerr=stds, color=colors, capsize=6,
                   alpha=0.85, edgecolor='k', linewidth=0.7)
axes[0].set_ylabel("Mean Episode Reward")
axes[0].set_title("Stage 3 — Mean Reward (± std)")
for bar, v in zip(bars, means):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.5, f"{v:.1f}",
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

bars2 = axes[1].bar(labels, surv, color=colors, alpha=0.85, edgecolor='k', linewidth=0.7)
axes[1].set_ylabel("Survival Rate (%)")
axes[1].set_ylim(0, 115)
axes[1].set_title("Stage 3 — Crash-free Episodes (%)")
for bar, v in zip(bars2, surv):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 1.5, f"{v:.0f}%",
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle("AVERT — Curriculum vs Scratch on Stage 3", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "stage3_head2head.png"), dpi=150, bbox_inches='tight')
plt.show()

# Per-episode scatter
fig, ax = plt.subplots(figsize=(11, 4))
ep_ids = range(1, N_EVAL + 1)
ax.plot(ep_ids, stats_curr['rewards'],    'o-', color='steelblue', label='Curriculum', alpha=0.85)
ax.plot(ep_ids, stats_scratch['rewards'], 's-', color='crimson',   label='Scratch',    alpha=0.85)
for i, (cr, sr) in enumerate(zip(stats_curr['crashed'], stats_scratch['crashed']), start=1):
    if cr:
        ax.scatter(i, stats_curr['rewards'][i-1],    marker='x', s=120, color='steelblue', zorder=5, linewidths=2.5)
    if sr:
        ax.scatter(i, stats_scratch['rewards'][i-1], marker='x', s=120, color='crimson',   zorder=5, linewidths=2.5)
ax.axhline(stats_curr['mean_reward'],    linestyle='--', color='steelblue', alpha=0.5,
           label=f"Curr mean ({stats_curr['mean_reward']:.1f})")
ax.axhline(stats_scratch['mean_reward'], linestyle='--', color='crimson',   alpha=0.5,
           label=f"Scratch mean ({stats_scratch['mean_reward']:.1f})")
ax.set_xlabel("Episode"); ax.set_ylabel("Episode Reward")
ax.set_title("Stage 3 — Per-episode Rewards  (✕ = crash)")
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "stage3_per_episode.png"), dpi=150)
plt.show()

In [ ]:
# ── GIF renders: Stage 3, both agents ────────────────────────────
print("\n=== Curriculum Agent — Stage 3 ===")
render_agent(model_curriculum, stage3_config, env_id="highway-v0",
             filename="curriculum_stage3.gif", n_episodes=3)

print("\n=== Scratch Agent — Stage 3 ===")
render_agent(model_scratch, stage3_config, env_id="highway-v0",
             filename="scratch_stage3.gif", n_episodes=3)

---
## 3 · Generalization: Medium-Difficulty Highway Config
**Medium config**: 40 PursuerVehicles, collision_reward = –3.  
Sits between Stage 2 and Stage 3 in difficulty: same hostile behavior type as Stage 3  
but 20% fewer cars and a 40% softer collision penalty.  
Tests whether the curriculum advantage holds — or whether scratch narrows the gap on easier ground.

In [ ]:
# ── Quantitative eval on medium config ───────────────────────────
print(f"Evaluating on Medium config ({N_EVAL} episodes each)...")
stats_curr_med    = eval_agent(model_curriculum, medium_config, n_episodes=N_EVAL)
stats_scratch_med = eval_agent(model_scratch,    medium_config, n_episodes=N_EVAL)

gap_s3  = stats_curr['mean_reward']     - stats_scratch['mean_reward']
gap_med = stats_curr_med['mean_reward'] - stats_scratch_med['mean_reward']

print("\n── Medium Config Results ────────────────────────")
for label, s in [("Curriculum", stats_curr_med), ("Scratch", stats_scratch_med)]:
    print(f"  {label:12s}  mean={s['mean_reward']:6.2f}  std={s['std_reward']:5.2f}"
          f"  survival={s['survival_rate']*100:.0f}%")
print(f"\n  Reward gap (Curriculum − Scratch):")
print(f"    Stage 3 : {gap_s3:+.2f}")
print(f"    Medium  : {gap_med:+.2f}")

In [ ]:
# ── Grouped bar: Stage 3 vs Medium ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x, w = np.arange(2), 0.35
xticks = ["Stage 3\n(50 Pursuers, –5)", "Medium\n(40 Pursuers, –3)"]

b1 = axes[0].bar(x - w/2, [stats_curr['mean_reward'],    stats_curr_med['mean_reward']],    w,
                 yerr=[stats_curr['std_reward'], stats_curr_med['std_reward']],
                 label='Curriculum', color='steelblue', capsize=5, alpha=0.85, edgecolor='k', linewidth=0.6)
b2 = axes[0].bar(x + w/2, [stats_scratch['mean_reward'], stats_scratch_med['mean_reward']], w,
                 yerr=[stats_scratch['std_reward'], stats_scratch_med['std_reward']],
                 label='Scratch', color='crimson', capsize=5, alpha=0.85, edgecolor='k', linewidth=0.6)
axes[0].set_xticks(x); axes[0].set_xticklabels(xticks)
axes[0].set_ylabel("Mean Episode Reward")
axes[0].set_title("Mean Reward — Stage 3 vs Medium")
axes[0].legend()
for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.3, f"{h:.1f}",
                     ha='center', va='bottom', fontsize=8)

sv_vals = [[stats_curr['survival_rate']*100,    stats_curr_med['survival_rate']*100],
           [stats_scratch['survival_rate']*100, stats_scratch_med['survival_rate']*100]]
b3 = axes[1].bar(x - w/2, sv_vals[0], w, label='Curriculum', color='steelblue', alpha=0.85, edgecolor='k', linewidth=0.6)
b4 = axes[1].bar(x + w/2, sv_vals[1], w, label='Scratch',    color='crimson',   alpha=0.85, edgecolor='k', linewidth=0.6)
axes[1].set_xticks(x); axes[1].set_xticklabels(xticks)
axes[1].set_ylim(0, 120)
axes[1].set_ylabel("Survival Rate (%)")
axes[1].set_title("Survival Rate — Stage 3 vs Medium")
axes[1].legend()
for bars in [b3, b4]:
    for bar in bars:
        h = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2, h + 1, f"{h:.0f}%",
                     ha='center', va='bottom', fontsize=8)

plt.suptitle("AVERT — Curriculum vs Scratch: Stage 3 and Medium Highway Config",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "medium_config_comparison.png"), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── GIF renders: medium config, both agents ───────────────────────
print("\n=== Curriculum Agent — Medium Config (40 Pursuers, –3) ===")
render_agent(model_curriculum, medium_config, env_id="highway-v0",
             filename="curriculum_medium.gif", n_episodes=3)

print("\n=== Scratch Agent — Medium Config (40 Pursuers, –3) ===")
render_agent(model_scratch, medium_config, env_id="highway-v0",
             filename="scratch_medium.gif", n_episodes=3)

---
## 4 · Training Efficiency Analysis
Reward histories saved during training reveal *how fast* each agent learned —  
independent of final performance.

In [ ]:
# ── Load reward histories ─────────────────────────────────────────
r_s1          = np.load(os.path.join(SAVE_DIR, "avert_stage1_rewards.npy"))
r_s2          = np.load(os.path.join(SAVE_DIR, "avert_stage2_rewards.npy"))
r_s3_curr     = np.load(os.path.join(SAVE_DIR, "avert_stage3_rewards.npy"))
r_s3_scratch  = np.load(os.path.join(SAVE_DIR, "avert_stage3_scratch_rewards.npy"))
r_rt          = np.load(os.path.join(SAVE_DIR, "avert_roundabout_transfer_rewards.npy"))
r_mt          = np.load(os.path.join(SAVE_DIR, "avert_merge_transfer_rewards.npy"))
r_rs          = np.load(os.path.join(SAVE_DIR, "avert_roundabout_scratch_rewards.npy"))
r_ms          = np.load(os.path.join(SAVE_DIR, "avert_merge_scratch_rewards.npy"))

print("Episodes per phase:")
for name, arr in [("Stage 1", r_s1), ("Stage 2", r_s2),
                  ("Stage 3 (curriculum)", r_s3_curr), ("Stage 3 (scratch)", r_s3_scratch),
                  ("Roundabout transfer", r_rt), ("Merge transfer", r_mt),
                  ("Roundabout scratch",  r_rs), ("Merge scratch",   r_ms)]:
    print(f"  {name:<26s}  {len(arr):>4d} ep   mean={np.mean(arr):.2f}")

In [ ]:
# ── Curriculum stage-by-stage progression ────────────────────────
SMOOTH_W = 5
stage_data = [
    ("Stage 1 — Passive IDM",  r_s1,      "steelblue"),
    ("Stage 2 — Aggressive",   r_s2,      "darkorange"),
    ("Stage 3 — Pursuers",     r_s3_curr, "crimson"),
]

fig, ax = plt.subplots(figsize=(13, 4))
offset = 0
for label, rewards, color in stage_data:
    xs = np.arange(offset, offset + len(rewards))
    ax.plot(xs, rewards, alpha=0.22, color=color, linewidth=0.8)
    s = smooth(rewards, SMOOTH_W)
    xs_sm = np.arange(offset + SMOOTH_W//2, offset + SMOOTH_W//2 + len(s))
    ax.plot(xs_sm, s, color=color, linewidth=2, label=label)
    ax.axvline(x=offset, linestyle='--', color=color, alpha=0.35)
    offset += len(rewards)

ax.set_xlabel("Episode (cumulative across stages)")
ax.set_ylabel("Episode Reward")
ax.set_title("AVERT — Curriculum Reward Progression Across All Stages")
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "curriculum_progression.png"), dpi=150)
plt.show()

In [ ]:
# ── Stage 3 convergence: curriculum phase vs scratch ──────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for rewards, label, color in [
    (r_s3_curr,   "Curriculum (Stage 3 phase)", "steelblue"),
    (r_s3_scratch, "Scratch",                   "crimson"),
]:
    axes[0].plot(rewards, alpha=0.2, color=color)
    axes[0].plot(smooth(rewards, SMOOTH_W), color=color, linewidth=2, label=label)
    cum = np.cumsum(rewards) / (np.arange(len(rewards)) + 1)
    axes[1].plot(cum, color=color, linewidth=2, label=label)

axes[0].set_xlabel("Episode"); axes[0].set_ylabel("Reward")
axes[0].set_title("Stage 3 Phase: Raw Learning Curves")
axes[0].legend()
axes[1].set_xlabel("Episode"); axes[1].set_ylabel("Cumulative Mean Reward")
axes[1].set_title("Stage 3 Phase: Convergence Speed")
axes[1].legend()

plt.suptitle("AVERT — Training Efficiency: Curriculum Stage 3 Phase vs Scratch",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "convergence_comparison.png"), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sample-efficiency: episodes to reach 90% of curriculum peak ──
def episodes_to_threshold(rewards, threshold, window=5):
    for i in range(window, len(rewards) + 1):
        if np.mean(rewards[max(0, i-window):i]) >= threshold:
            return i
    return None

curr_peak    = np.mean(r_s3_curr[-max(1, len(r_s3_curr)//5):])
scratch_peak = np.mean(r_s3_scratch[-max(1, len(r_s3_scratch)//5):])
threshold    = curr_peak * 0.9

ep_curr    = episodes_to_threshold(r_s3_curr,    threshold)
ep_scratch = episodes_to_threshold(r_s3_scratch, threshold)

print("── Sample Efficiency (Stage 3) ─────────────────")
print(f"  Curriculum final mean (last 20%): {curr_peak:.2f}")
print(f"  Scratch    final mean (last 20%): {scratch_peak:.2f}")
print(f"  Target threshold (90% of curriculum peak): {threshold:.2f}")
print(f"  Curriculum episodes to reach threshold: {ep_curr}")
print(f"  Scratch    episodes to reach threshold: {ep_scratch}")
if ep_curr and ep_scratch:
    print(f"  Curriculum was {ep_scratch/ep_curr:.1f}x faster to reach threshold")

---
## 5 · Transfer Learning: Curriculum Transfer vs Scratch
The fully-trained Stage 3 model was fine-tuned on roundabout and merge environments.  
Scratch baselines were also trained from scratch on those same environments.

In [ ]:
# ── Transfer learning curves ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, r_transfer, r_scratch, label in [
    (axes[0], r_rt, r_rs, "Roundabout"),
    (axes[1], r_mt, r_ms, "Merge"),
]:
    ax.plot(r_transfer, alpha=0.22, color='purple')
    ax.plot(smooth(r_transfer, SMOOTH_W), color='purple', linewidth=2,
            label=f'Curriculum Transfer (mean={np.mean(r_transfer):.2f})')
    ax.plot(r_scratch, alpha=0.22, color='seagreen')
    ax.plot(smooth(r_scratch, SMOOTH_W), color='seagreen', linewidth=2,
            label=f'Scratch (mean={np.mean(r_scratch):.2f})')
    ax.set_xlabel("Episode"); ax.set_ylabel("Episode Reward")
    ax.set_title(f"{label} — Transfer vs Scratch")
    ax.legend(fontsize=8)

plt.suptitle("AVERT — Transfer Learning: Curriculum Stage 3 → New Env vs Scratch",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "transfer_vs_scratch.png"), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Transfer bar chart ────────────────────────────────────────────
xlabels    = ["Roundabout\nTransfer", "Roundabout\nScratch",
              "Merge\nTransfer",      "Merge\nScratch"]
vals       = [np.mean(r_rt), np.mean(r_rs), np.mean(r_mt), np.mean(r_ms)]
stds_t     = [np.std(r_rt),  np.std(r_rs),  np.std(r_mt),  np.std(r_ms)]
bar_colors = ['purple', 'seagreen', 'purple', 'seagreen']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(np.arange(4), vals, yerr=stds_t, color=bar_colors,
              alpha=0.85, capsize=5, edgecolor='k', linewidth=0.6)
ax.set_xticks(np.arange(4)); ax.set_xticklabels(xlabels)
ax.set_ylabel("Mean Episode Reward")
ax.set_title("AVERT — Transfer Learning: Curriculum Transfer vs Scratch")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.2, f"{v:.1f}",
            ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.legend(handles=[
    mpatches.Patch(color='purple',   alpha=0.85, label='Curriculum Transfer'),
    mpatches.Patch(color='seagreen', alpha=0.85, label='Scratch'),
])
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "transfer_bar.png"), dpi=150)
plt.show()

In [ ]:
# ── GIF renders: Roundabout — transfer vs scratch ─────────────────
print("\n=== Curriculum Transfer Agent — Roundabout ===")
render_agent(model_roundabout_transfer, roundabout_config, env_id="roundabout-v0",
             filename="curriculum_roundabout.gif", n_episodes=3)

print("\n=== Scratch Agent (trained on Roundabout) — Roundabout ===")
render_agent(model_roundabout_scratch, roundabout_config, env_id="roundabout-v0",
             filename="scratch_roundabout.gif", n_episodes=3)

In [ ]:
# ── GIF renders: Merge — transfer vs scratch ──────────────────────
print("\n=== Curriculum Transfer Agent — Merge ===")
render_agent(model_merge_transfer, merge_config, env_id="merge-v0",
             filename="curriculum_merge.gif", n_episodes=3)

print("\n=== Scratch Agent (trained on Merge) — Merge ===")
render_agent(model_merge_scratch, merge_config, env_id="merge-v0",
             filename="scratch_merge.gif", n_episodes=3)

In [ ]:
# ── GIF renders: Merge — transfer vs scratch ──────────────────────
print("\n=== Curriculum Transfer Agent — Merge ===")
render_agent(model_merge_transfer, merge_config, env_id="merge-v0",
             filename="curriculum_merge.gif", n_episodes=3)

print("\n=== Scratch Agent — Merge ===")
render_agent(model_scratch, merge_config, env_id="merge-v0",
             filename="scratch_merge.gif", n_episodes=3)

---
## 6 · Full Summary Dashboard

In [ ]:
# ── 2×2 summary figure ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("AVERT — Full Experiment Summary", fontsize=14, fontweight='bold')

# (0,0) Stage 3 bar
ax = axes[0, 0]
ax.bar(["Curriculum", "Scratch"],
       [stats_curr['mean_reward'], stats_scratch['mean_reward']],
       yerr=[stats_curr['std_reward'], stats_scratch['std_reward']],
       color=['steelblue', 'crimson'], alpha=0.85, capsize=5, edgecolor='k', linewidth=0.6)
ax.set_title("Stage 3 — Mean Reward", fontweight='bold')
ax.set_ylabel("Mean Episode Reward")

# (0,1) Stage 3 vs Medium grouped
ax = axes[0, 1]
x, w = np.arange(2), 0.32
ax.bar(x - w/2, [stats_curr['mean_reward'],    stats_curr_med['mean_reward']],
       w, color='steelblue', alpha=0.85, edgecolor='k', linewidth=0.6, label='Curriculum')
ax.bar(x + w/2, [stats_scratch['mean_reward'], stats_scratch_med['mean_reward']],
       w, color='crimson', alpha=0.85, edgecolor='k', linewidth=0.6, label='Scratch')
ax.set_xticks(x); ax.set_xticklabels(["Stage 3", "Medium"])
ax.set_title("Generalization: Stage 3 → Medium Config", fontweight='bold')
ax.set_ylabel("Mean Reward"); ax.legend(fontsize=8)

# (1,0) Curriculum progression
ax = axes[1, 0]
offset = 0
for label, rewards, color in stage_data:
    xs = np.arange(offset, offset + len(rewards))
    ax.plot(xs, rewards, alpha=0.15, color=color)
    s = smooth(rewards, SMOOTH_W)
    xs_sm = np.arange(offset + SMOOTH_W//2, offset + SMOOTH_W//2 + len(s))
    ax.plot(xs_sm, s, color=color, linewidth=2, label=label)
    ax.axvline(x=offset, linestyle='--', color=color, alpha=0.3)
    offset += len(rewards)
ax.set_xlabel("Episode (cumulative)"); ax.set_ylabel("Reward")
ax.set_title("Curriculum Stage Progression", fontweight='bold')
ax.legend(fontsize=7)

# (1,1) Transfer bar
ax = axes[1, 1]
ax.bar(np.arange(4), vals, yerr=stds_t,
       color=['purple','seagreen','purple','seagreen'],
       alpha=0.85, capsize=4, edgecolor='k', linewidth=0.6)
ax.set_xticks(np.arange(4))
ax.set_xticklabels(["RB Transfer", "RB Scratch", "Merge Transfer", "Merge Scratch"], fontsize=8)
ax.set_ylabel("Mean Reward")
ax.set_title("Transfer Learning vs Scratch", fontweight='bold')
ax.legend(handles=[
    mpatches.Patch(color='purple',   alpha=0.85, label='Transfer'),
    mpatches.Patch(color='seagreen', alpha=0.85, label='Scratch'),
], fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "avert_summary_dashboard.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved avert_summary_dashboard.png")

In [ ]:
# ── Printed results table ─────────────────────────────────────────
print("=" * 65)
print(f"{'AVERT — FULL RESULTS TABLE':^65}")
print("=" * 65)
rows = [
    ("Stage 3 (Curriculum)",       stats_curr['mean_reward'],       stats_curr['std_reward'],       stats_curr['survival_rate']),
    ("Stage 3 (Scratch)",          stats_scratch['mean_reward'],    stats_scratch['std_reward'],    stats_scratch['survival_rate']),
    ("Medium Config (Curriculum)", stats_curr_med['mean_reward'],   stats_curr_med['std_reward'],   stats_curr_med['survival_rate']),
    ("Medium Config (Scratch)",    stats_scratch_med['mean_reward'],stats_scratch_med['std_reward'],stats_scratch_med['survival_rate']),
    ("Roundabout (Transfer)",      np.mean(r_rt), np.std(r_rt),    None),
    ("Roundabout (Scratch)",       np.mean(r_rs), np.std(r_rs),    None),
    ("Merge (Transfer)",           np.mean(r_mt), np.std(r_mt),    None),
    ("Merge (Scratch)",            np.mean(r_ms), np.std(r_ms),    None),
]
print(f"  {'Condition':<30s} {'Mean R':>7s} {'Std':>6s} {'Surv%':>7s}")
print("  " + "-" * 55)
for name, mean, std, surv in rows:
    surv_str = f"{surv*100:.0f}%" if surv is not None else "  n/a"
    print(f"  {name:<30s} {mean:>7.2f} {std:>6.2f} {surv_str:>7s}")
print("=" * 65)
print(f"  Curriculum advantage on Stage 3   : {gap_s3:+.2f}")
print(f"  Curriculum advantage on Medium    : {gap_med:+.2f}")
print(f"  Roundabout: transfer vs scratch   : {np.mean(r_rt)-np.mean(r_rs):+.2f}")
print(f"  Merge:      transfer vs scratch   : {np.mean(r_mt)-np.mean(r_ms):+.2f}")